# Zero-Shot Generalization with Random Forest on Titanic

Zero-shot learning classifies data from **unseen distributions** without task-specific retraining. In this notebook you will:
- Train a RandomForestClassifier on one passenger subgroup
- Apply it to an entirely different subgroup without retraining
- Compare zero-shot generalization vs. supervised training on the target group

## Problem Definition

Zero-shot learning advantages:
- **No labeled data required** for the new target distribution
- **Quick adaptation** to new passenger subgroups
- **Resource-efficient**: no retraining needed

For Titanic: train on first/second class passengers, apply to third class — a distribution shift scenario.

In [ ]:
import numpy as np, pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings; warnings.filterwarnings('ignore')

## Prepare Dataset

In [ ]:
def load_titanic_raw():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    return df

df = load_titanic_raw()

# Source domain: 1st and 2nd class passengers (seen distribution)
source = df[df['pclass'].isin([1, 2])].copy()

# Target domain: 3rd class passengers (unseen distribution — zero-shot)
target = df[df['pclass'] == 3].copy()

X_source = source.drop('survived', axis=1)
y_source = source['survived']
X_target = target.drop('survived', axis=1)
y_target = target['survived']

print(f"Source (1st+2nd class): {len(X_source)} passengers, survival rate: {y_source.mean():.2%}")
print(f"Target (3rd class):     {len(X_target)} passengers, survival rate: {y_target.mean():.2%}")

## Zero-Shot Classifier

Train on the source domain (1st+2nd class) and apply directly to the target domain (3rd class) without any target-domain training examples.

In [ ]:
class ZeroShotRFClassifier:
    '''
    Zero-shot classifier using feature semantics.
    Trained on source domain, applied to target domain without retraining.
    '''
    def __init__(self, n_estimators=100, random_state=42):
        self.model = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state)
        self.scaler = StandardScaler()

    def create_prompt(self, X, description):
        '''Describe the dataset — analogous to zero-shot prompt engineering.'''
        print(f"\nApplying zero-shot to: {description}")
        print(f"  Samples: {len(X)}")
        print(f"  Features: {list(X.columns)}")
        return X

    def fit(self, X_train, y_train):
        X_s = self.scaler.fit_transform(X_train)
        self.model.fit(X_s, y_train)

    def generate_response(self, X_target, description):
        '''Generate predictions for target domain — analogous to LLM zero-shot response.'''
        X_prompted = self.create_prompt(X_target, description)
        X_s = self.scaler.transform(X_prompted)
        return self.model.predict(X_s)

## Train on Source Domain (Seen Classes)

In [ ]:
# Train zero-shot classifier on source domain (1st + 2nd class)
zero_shot = ZeroShotRFClassifier(n_estimators=100, random_state=42)
zero_shot.fit(X_source, y_source)

# Source domain accuracy (in-distribution)
y_source_pred = zero_shot.generate_response(X_source, "Source: 1st+2nd class (in-distribution)")
source_acc = accuracy_score(y_source, y_source_pred)
print(f"\nSource domain accuracy: {source_acc:.4f}")

## Zero-Shot Classification on Unseen Target Domain

In [ ]:
# Zero-shot: apply to 3rd class passengers WITHOUT any retraining
y_zero_shot_pred = zero_shot.generate_response(X_target, "Target: 3rd class (zero-shot, unseen distribution)")
zero_shot_acc = accuracy_score(y_target, y_zero_shot_pred)
print(f"Zero-shot accuracy on 3rd class: {zero_shot_acc:.4f}")
print("\nZero-shot Classification Report:")
print(classification_report(y_target, y_zero_shot_pred, target_names=['Not Survived','Survived']))

## Comparison: Zero-Shot vs. Supervised Training on Target

In [ ]:
# Supervised: train directly on 3rd class data (train/test split)
X_tgt_tr, X_tgt_te, y_tgt_tr, y_tgt_te = train_test_split(
    X_target, y_target, test_size=0.3, stratify=y_target, random_state=42)
scaler_sup = StandardScaler().fit(X_tgt_tr)

supervised = RandomForestClassifier(n_estimators=100, random_state=42)
supervised.fit(scaler_sup.transform(X_tgt_tr), y_tgt_tr)
supervised_acc = accuracy_score(y_tgt_te, supervised.predict(scaler_sup.transform(X_tgt_te)))
print(f"Supervised (3rd class only): {supervised_acc:.4f}")

# Zero-shot on same test split
zero_shot_te_acc = accuracy_score(y_tgt_te,
    zero_shot.model.predict(zero_shot.scaler.transform(X_tgt_te)))
print(f"Zero-shot  (3rd class):      {zero_shot_te_acc:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
labels = ['Zero-Shot\n(no 3rd-class data)', 'Supervised\n(3rd-class training data)']
accs   = [zero_shot_te_acc, supervised_acc]
colors = ['#FF9800','#2196F3']
bars = ax.bar(labels, accs, color=colors, edgecolor='white', linewidth=2)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{acc:.3f}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0,1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Zero-Shot vs Supervised — 3rd Class Passengers')
plt.tight_layout()
plt.show()

Zero-shot learning allows classification of unseen passenger groups without any labeled examples from that group. The feature semantics learned from 1st+2nd class generalize to 3rd class, demonstrating cross-distribution generalization.